# Exercises 2.1 and 2.2: Trip production and attraction (4 points): Person A

Report chapter: `report/chapters/04_trip_production.tex and 05_trip_attraction.tex`

**Sub-questions**

1. Ex 2.1: Total number of trips (1 point)
2. Ex 2.1: Trips per SPC and map of trips per IRIS (1 point)
3. Ex 2.2: Total N and arriving trips per municipality (2 points)
4. Ex 2.2: Scaling of O_i and hand-off file

**Reads**

- data/raw/ (aggregated census IRIS CSV)
- data/processed/employment.parquet
- data/processed/iris.gpkg
- data/processed/municipalities.gpkg

**Writes**

- data/processed/zones.parquet  (HAND-OFF to Person B, contract in the setup cell)
- report/figures/trips_per_spc.pdf
- report/figures/trips_per_iris_map.pdf
- report/figures/arrivals_map.pdf

## Setup and configuration

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt

# ---- Study area ------------------------------------------------------------------------
DEPARTMENT = "14"            # INSEE code of Calvados (region Normandie)
CRS = "EPSG:2154"            # RGF93 / Lambert-93, metres; all geometries and distances

# ---- Paths -----------------------------------------------------------------------------
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir() and (p / "report").is_dir())
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
FIGURES_DIR = ROOT / "report" / "figures"

# TODO: adjust the file name to what was downloaded (data/DATA_SOURCES.md)
CENSUS_AGGREGATED_CSV = DATA_RAW / "TODO_base-ic-evol-struct-pop-2019.csv"   # n_{i,s} = C19_POP15P_CS1..8 (verify)
IRIS_GPKG = DATA_PROCESSED / "iris.gpkg"                                      # from ex11
MUNICIPALITIES_GPKG = DATA_PROCESSED / "municipalities.gpkg"                  # from ex11
EMPLOYMENT_PARQUET = DATA_PROCESSED / "employment.parquet"                    # from ex13
ZONES_PARQUET = DATA_PROCESSED / "zones.parquet"                              # HAND-OFF A -> B
ZONES_PLACEHOLDER_MARKER = DATA_PROCESSED / "zones.PLACEHOLDER"

# ---- Model parameters (assignment, Autumn 2026) -----------------------------------------
# Exercise 2.1, Table 1: O_i = max(beta_0 + sum_s beta_s * n_{i,s}, 0). The offset is added ONCE.
TRIP_GEN_OFFSET = 27.244
TRIP_GEN_BETA = {1: 0.319, 2: 0.994, 3: 0.863, 4: 0.990, 5: 0.780, 6: 0.708, 7: 0.120, 8: -0.073}
# Exercise 2.2: N = min(0.7 * sum_j W_j, 0.7 * sum_i O_i)
COMMUTER_SHARE = 0.7

SPC_LABELS = {
    1: "Agriculteurs exploitants",
    2: "Artisans, commerçants et chefs d'entreprise",
    3: "Cadres et professions intellectuelles supérieures",
    4: "Professions intermédiaires",
    5: "Employés",
    6: "Ouvriers",
    7: "Retraités",
    8: "Autres personnes sans activité professionnelle",
}

# ---- Hand-off file zones.parquet (ex21_22 writes, ex23-ex32 read) --------------------------------
# One row per municipality: municipality_id (str, unique), name (str), geometry (EPSG:2154),
# O (float >= 0, scaled trips O'_i), D (float >= 0, arriving trips D_j), with O.sum() == D.sum() == N.
# The row order fixes the zone order of all vectors and matrices in Person B's part.
ZONE_COLUMNS = ["municipality_id", "name", "geometry", "O", "D"]


def validate_zones(zones, rtol=1e-9):
    # Raise ValueError if the hand-off contract is violated
    missing = [c for c in ZONE_COLUMNS if c not in zones.columns]
    if missing:
        raise ValueError(f"zones is missing columns: {missing}")
    if zones.crs is None or not zones.crs.equals(CRS):
        raise ValueError(f"zones must have CRS {CRS}, got {zones.crs}")
    if zones["municipality_id"].duplicated().any():
        raise ValueError("municipality_id must be unique")
    if zones[["O", "D"]].isna().any().any() or (zones[["O", "D"]] < 0).any().any():
        raise ValueError("O and D must be non-negative and not NaN")
    if not np.isclose(zones["O"].sum(), zones["D"].sum(), rtol=rtol, atol=0.0):
        raise ValueError(f"O.sum() != D.sum(): {zones['O'].sum()} vs {zones['D'].sum()} (both must equal N)")


def write_zones(zones, placeholder=False):
    # Validate and write zones.parquet; placeholder=True creates the marker file, a real write removes it
    validate_zones(zones)
    zones[ZONE_COLUMNS].to_parquet(ZONES_PARQUET)
    if placeholder:
        ZONES_PLACEHOLDER_MARKER.write_text(
            "zones.parquet contains PLACEHOLDER data (O = population 15+, D = URSSAF employment, "
            "scaled to the same total). Not the result of Exercises 2.1/2.2.\n"
        )
    else:
        ZONES_PLACEHOLDER_MARKER.unlink(missing_ok=True)

# ---- Figures ---------------------------------------------------------------------------
mpl.rcParams.update({
    "figure.dpi": 100, "savefig.dpi": 300, "font.size": 9, "axes.titlesize": 10,
    "axes.labelsize": 9, "legend.fontsize": 8, "pdf.fonttype": 42,
})


def style_map_axes(ax):
    # Common map look: equal aspect, no axes frame or ticks
    ax.set_aspect("equal")
    ax.set_axis_off()
    return ax


def save_figure(fig, name, preview_png=False):
    # Vector PDF for the report (report/figures/<name>.pdf, \includegraphics{<name>}), optional PNG preview
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    path = FIGURES_DIR / f"{name}.pdf"
    fig.savefig(path, format="pdf", bbox_inches="tight")
    if preview_png:
        fig.savefig(path.with_suffix(".png"), format="png", dpi=150, bbox_inches="tight")
    return path

## 1. Ex 2.1: Total number of trips (1 point)

Apply `O_i = max(beta_0 + sum_s beta_s * n_is, 0)` with the parameters of the setup cell (Table 1). **The offset beta_0 is added once**, not once per SPC. Decide on which zone the model is evaluated (IRIS or municipality; the offset and the max make the total depend on it) and document it. Report the total number of trips.

In [ ]:
# TODO

## 2. Ex 2.1: Trips per SPC and map of trips per IRIS (1 point)

Bar plot of the trips generated per SPC, and a map of the total number of generated trips per IRIS. Figures: `trips_per_spc`, `trips_per_iris_map`.

In [ ]:
# TODO

## 3. Ex 2.2: Total N and arriving trips per municipality (2 points)

`N = min(0.7 * sum_j W_j, 0.7 * sum_i O_i)`, `D_j = W_j / sum W * N`. Report N and map the arriving trips per municipality. Figure: `arrivals_map`.

In [ ]:
# TODO

## 4. Ex 2.2: Scaling of O_i and hand-off file

Scale `O'_i = N * O_i / sum O` so that `O.sum() == D.sum() == N`, build the municipality `GeoDataFrame` with columns `municipality_id, name, geometry, O, D` and write it with `write_zones(zones)` (setup cell), which validates the contract and removes the placeholder marker.

In [ ]:
# TODO

## Problems encountered

TODO: note every problem and how it was solved (partial points are given for documented problems); copy it into the `\paragraph{Problems encountered.}` of the report chapter.